In [3]:
import numpy as np
import plotly.graph_objects as go

# -----------------------
# Generate data
# -----------------------
np.random.seed(42)

n = 700
mean = [0, 0, 0]
cov = [[1.5, 1.2, 1.0],
       [1.2, 1.5, 1.1],
       [1.0, 1.1, 1.5]]

X = np.random.multivariate_normal(mean, cov, n)

# Color groups (to mimic gradient clusters)
colors = np.where(X[:, 0] > 1.2, '#d627ff',   # magenta
         np.where(X[:, 0] > 0.3, '#17becf',   # cyan
         np.where(X[:, 0] > -0.8, '#2ca02c',  # green
                                   '#ffbf00'))) # yellow/orange

# -----------------------
# PCA for arrows
# -----------------------
Xc = X - X.mean(axis=0)
_, _, Vt = np.linalg.svd(Xc, full_matrices=False)
components = Vt[:3]

origin = X.mean(axis=0)
scale = 4

arrow_colors = ['blue', 'green', 'red']

# -----------------------
# Scatter
# -----------------------
scatter = go.Scatter3d(
    x=X[:, 0],
    y=X[:, 1],
    z=X[:, 2],
    mode='markers',
    marker=dict(size=3, color=colors, opacity=0.9)
)

# -----------------------
# Arrows (lines + cones)
# -----------------------
arrows = []
for i in range(3):
    vec = components[i] * scale

    # line
    arrows.append(go.Scatter3d(
        x=[origin[0], origin[0] + vec[0]],
        y=[origin[1], origin[1] + vec[1]],
        z=[origin[2], origin[2] + vec[2]],
        mode='lines',
        line=dict(color=arrow_colors[i], width=6),
        showlegend=False
    ))

    # cone tip
    arrows.append(go.Cone(
        x=[origin[0] + vec[0]],
        y=[origin[1] + vec[1]],
        z=[origin[2] + vec[2]],
        u=[vec[0]],
        v=[vec[1]],
        w=[vec[2]],
        anchor="tip",
        showscale=False,
        colorscale=[[0, arrow_colors[i]], [1, arrow_colors[i]]],
        sizemode="absolute",
        sizeref=0.6
    ))

# -----------------------
# Layout (match grey bg)
# -----------------------
fig = go.Figure(data=[scatter] + arrows)

fig.update_layout(
    margin=dict(l=0, r=0, b=0, t=0),
    scene=dict(
        xaxis_title='',
        yaxis_title='',
        zaxis_title='',

        # light grey panels like your image
        xaxis=dict(backgroundcolor='rgb(240,240,240)', gridcolor='white'),
        yaxis=dict(backgroundcolor='rgb(240,240,240)', gridcolor='white'),
        zaxis=dict(backgroundcolor='rgb(240,240,240)', gridcolor='white'),
    ),
    paper_bgcolor='white'
)

fig.show()